In [21]:
%%sql
-- DROP TABLE dbo.fact_sod_conflict;

CREATE OR REPLACE TABLE fact_sod_conflict AS
SELECT
    assesment_id as assessment_record_id,
    review_id,
    user_id,
    risk_id,
    system_id,
    conflict_detection_date
FROM dbo.silver_sod_risk_assessment;

StatementMeta(, da1dd069-4ed7-415b-9403-1239554a3ea8, 30, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [22]:
SELECT * FROM lh_SAP_IAM_Data_Analysis.dbo.dim_risk LIMIT 1000

StatementMeta(, da1dd069-4ed7-415b-9403-1239554a3ea8, 31, Finished, Available, Finished, False)

<Spark SQL result set with 120 rows and 6 fields>

In [23]:
select * from dbo.fact_sod_conflict
limit 10;

StatementMeta(, da1dd069-4ed7-415b-9403-1239554a3ea8, 32, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 6 fields>

In [24]:
CREATE OR REPLACE TABLE bridge_conflict_role AS
SELECT 
    f.assessment_record_id,
    r.role_key
FROM dbo.fact_sod_conflict f
JOIN dbo.silver_role_conflict_assignment r
    ON f.assessment_record_id = r.assessment_record_id;  


StatementMeta(, da1dd069-4ed7-415b-9403-1239554a3ea8, 33, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [25]:
-- Checks 
SELECT COUNT(*) AS bridge_rows
FROM dbo.bridge_conflict_role;

StatementMeta(, da1dd069-4ed7-415b-9403-1239554a3ea8, 34, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [26]:
-- Checks 
SELECT
    assessment_record_id,
    role_key,
    COUNT(*) AS relationship_count
FROM dbo.bridge_conflict_role
GROUP BY
    assessment_record_id,
    role_key
HAVING COUNT(*) > 1;

StatementMeta(, da1dd069-4ed7-415b-9403-1239554a3ea8, 35, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 3 fields>

In [27]:
-- Checks 
SELECT b.role_key
FROM dbo.bridge_conflict_role b
LEFT JOIN dbo.silver_role_master r
    ON b.role_key = r.role_key
WHERE r.role_key IS NULL;

StatementMeta(, da1dd069-4ed7-415b-9403-1239554a3ea8, 36, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 1 fields>

In [28]:
-- Checks User
SELECT f.user_id
FROM dbo.fact_sod_conflict f
LEFT JOIN dbo.silver_user_master u
    ON f.user_id = u.user_id
WHERE u.user_id IS NULL;

-- Checks Risk
SELECT f.risk_id
FROM dbo.fact_sod_conflict f
LEFT JOIN dbo.silver_risk_master r
    ON f.risk_id = r.risk_id
WHERE r.risk_id IS NULL;

-- Checks System
SELECT f.system_id
FROM dbo.fact_sod_conflict f
LEFT JOIN dbo.silver_system_master s
    ON f.system_id = s.system_id
WHERE s.system_id IS NULL;

-- Checks Review
SELECT f.review_id
FROM dbo.fact_sod_conflict f
LEFT JOIN dbo.silver_risk_review_master r
    ON f.review_id = r.review_id
WHERE r.review_id IS NULL;

StatementMeta(, da1dd069-4ed7-415b-9403-1239554a3ea8, 40, Finished, Available, Finished, True)

<Spark SQL result set with 189 rows and 1 fields>

<Spark SQL result set with 188 rows and 1 fields>

<Spark SQL result set with 0 rows and 1 fields>

<Spark SQL result set with 0 rows and 1 fields>

In [29]:
CREATE OR REPLACE TABLE dim_risk AS
SELECT
    risk_id,
    risk_name,
    risk_level,
    business_process,
    function_1,
    function_2
FROM dbo.silver_risk_master;

SELECT * FROM dim_risk;

StatementMeta(, da1dd069-4ed7-415b-9403-1239554a3ea8, 42, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 120 rows and 6 fields>

In [30]:
CREATE OR REPLACE TABLE  dim_risk_review AS
SELECT
    review_id, 
    review_name, 
    review_date
FROM dbo.silver_risk_review_master;


SELECT * FROM dim_risk_review;

StatementMeta(, da1dd069-4ed7-415b-9403-1239554a3ea8, 43, Finished, , Finished, True)

Error: [TABLE_OR_VIEW_ALREADY_EXISTS] Cannot create table or view `chimcobldhq2atrjbt9k2k2v950kqnq4c5q62nq1dpgmoubjd5piar38bt9k2k2v950kqnq4c5q62nq1dpgmoubjd5piap32ds`.`dim_risk_review` because it already exists.
Choose a different name, drop or replace the existing object, or add the IF NOT EXISTS clause to tolerate pre-existing objects.

In [ ]:
-- DROP TABLE dbo.dim_role;
CREATE OR REPLACE TABLE dim_role AS
SELECT
    role_key,
    role_name,
    role_type,
    sap_module,
    risk_indicator
FROM dbo.silver_role_master;

SELECT * FROM dim_role;


StatementMeta(, da1dd069-4ed7-415b-9403-1239554a3ea8, -1, Cancelled, , Cancelled, True)

In [ ]:
CREATE OR REPLACE TABLE dim_system AS
SELECT
    system_id,
    system_name
FROM dbo.silver_system_master;

SELECT * FROM dim_system;

StatementMeta(, da1dd069-4ed7-415b-9403-1239554a3ea8, -1, Cancelled, , Cancelled, True)

In [ ]:
CREATE OR REPLACE TABLE dim_user AS
SELECT
    user_id,
	user_name,
	business_unit,
	department,
	region,
	active_status
FROM dbo.silver_user_master;


SELECT * FROM dim_user;

StatementMeta(, da1dd069-4ed7-415b-9403-1239554a3ea8, -1, Cancelled, , Cancelled, True)

In [ ]:
CREATE OR REPLACE TABLE fact_sod_conflict_lifecycle AS

WITH conflict_years AS
(
    SELECT DISTINCT
        f.user_id,
        f.risk_id,
        f.system_id,
        YEAR(r.review_date) AS review_year
    FROM dbo.fact_sod_conflict f
    INNER JOIN dbo.dim_risk_review r
        ON f.review_id = r.review_id
),

conflict_history AS
(
    SELECT
        user_id,
        risk_id,
        system_id,
        MIN(review_year) AS first_seen_year,
        MAX(review_year) AS last_seen_year,
        COUNT(DISTINCT review_year) AS review_count
    FROM conflict_years
    GROUP BY
        user_id,
        risk_id,
        system_id
),

annual_status AS
(
    SELECT
        c.user_id,
        c.risk_id,
        c.system_id,
        c.review_year,
        h.first_seen_year,
        h.last_seen_year,
        h.review_count,

        CASE
            WHEN c.review_year = h.first_seen_year
                THEN 'New'

            WHEN EXISTS
            (
                SELECT 1
                FROM conflict_years p
                WHERE p.user_id = c.user_id
                  AND p.risk_id = c.risk_id
                  AND p.system_id = c.system_id
                  AND p.review_year = c.review_year - 1
            )
                THEN 'Persistent'

            ELSE 'New'
        END AS lifecycle_status

    FROM conflict_years c
    INNER JOIN conflict_history h
        ON c.user_id = h.user_id
        AND c.risk_id = h.risk_id
        AND c.system_id = h.system_id
),

resolved_events AS
(
    SELECT
        p.user_id,
        p.risk_id,
        p.system_id,
        p.review_year AS review_year,
        h.first_seen_year,
        h.last_seen_year,
        h.review_count,
        'Resolved' AS lifecycle_status

    FROM conflict_years p

    INNER JOIN conflict_history h
        ON p.user_id = h.user_id
        AND p.risk_id = h.risk_id
        AND p.system_id = h.system_id

    WHERE NOT EXISTS
    (
        SELECT 1
        FROM conflict_years c
        WHERE c.user_id = p.user_id
          AND c.risk_id = p.risk_id
          AND c.system_id = p.system_id
          AND c.review_year = p.review_year + 1
    )
)

SELECT *
FROM annual_status

UNION ALL

SELECT *
FROM resolved_events;

StatementMeta(, da1dd069-4ed7-415b-9403-1239554a3ea8, -1, Cancelled, , Cancelled, True)

In [ ]:
select DISTInct COUNT(*) from dbo.fact_sod_conflict_lifecycle
where lifecycle_status  = "Resolved";

StatementMeta(, da1dd069-4ed7-415b-9403-1239554a3ea8, -1, Cancelled, , Cancelled, True)